In [1]:
import numpy as np
import pandas as pd

# =========================
# CONFIG (your confirmed specs)
# =========================
N_SAMPLES = 10000

V_MIN, V_MAX = 9.0, 12.0          # volts (3S pack)
I_MIN, I_MAX = -4.0, 4.0          # amps (charge/discharge)
T_MIN, T_MAX = 25.0, 45.0         # °C
CYCLE_MIN, CYCLE_MAX = 0, 300     # cycles
SOH_MIN, SOH_MAX = 90.0, 100.0    # %

# =========================
# GENERATE FEATURES
# =========================
np.random.seed(42)

voltage = np.random.uniform(V_MIN, V_MAX, N_SAMPLES)
current = np.random.uniform(I_MIN, I_MAX, N_SAMPLES)
temperature = np.random.uniform(T_MIN, T_MAX, N_SAMPLES)
cycle = np.random.randint(CYCLE_MIN, CYCLE_MAX+1, N_SAMPLES)

# =========================
# SOC MODEL (realistic)
# =========================
# base SOC from voltage curve
soc_base = (voltage - V_MIN) / (V_MAX - V_MIN) * 100

# current effect
soc_current_effect = -current * 2.5   # discharge lowers SOC reading

# temp effect
soc_temp_effect = (temperature - 25) * 0.2

SOC = soc_base + soc_current_effect + soc_temp_effect
SOC = np.clip(SOC, 0, 100)

# =========================
# SOH MODEL (cycle aging)
# =========================
SOH = SOH_MAX - (cycle / CYCLE_MAX) * (SOH_MAX - SOH_MIN)

# small noise
SOH += np.random.normal(0, 0.15, N_SAMPLES)
SOH = np.clip(SOH, SOH_MIN, SOH_MAX)

# =========================
# DATAFRAME
# =========================
df = pd.DataFrame({
    "voltage": voltage,
    "current": current,
    "temperature": temperature,
    "cycle": cycle,
    "SOC": SOC,
    "SOH": SOH
})

# =========================
# SAVE CSV
# =========================
csv_path = "synthetic_battery_3S.csv"
df.to_csv(csv_path, index=False)



In [2]:
# =========================
# DATASET INFO
# =========================
print("CSV saved:", csv_path)
print("\nShape (rows, cols):", df.shape)
print("\nColumn info:")
print(df.info())

print("\nStatistics:")
print(df.describe())

print("\nFirst 5 rows:")
print(df.head())

CSV saved: synthetic_battery_3S.csv

Shape (rows, cols): (10000, 6)

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   voltage      10000 non-null  float64
 1   current      10000 non-null  float64
 2   temperature  10000 non-null  float64
 3   cycle        10000 non-null  int64  
 4   SOC          10000 non-null  float64
 5   SOH          10000 non-null  float64
dtypes: float64(5), int64(1)
memory usage: 468.9 KB
None

Statistics:
            voltage       current   temperature         cycle           SOC  \
count  10000.000000  10000.000000  10000.000000  10000.000000  10000.000000   
mean      10.482479      0.036239     35.001008    151.556600     51.241509   
std        0.862890      2.314356      5.735475     87.476047     29.060086   
min        9.000035     -3.998738     25.000962      0.000000      0.000000   
25%       